# Silver — `ecommerce_pedidos`

Este notebook lê micro-lotes da Bronze, aplica as 10 regras de qualidade da tabela de pedidos, grava a Silver em Delta e registra os resultados em `squad1.dq_monitoring_logs`.



In [0]:
%run ../utils/utils

## Inicialização e Orquestração

In [0]:


import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_pedidos"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - Run ID: {RUN_ID}")

## Leitura Dinâmica do Micro-lote e Tabelas de Referência

In [0]:
# 1. Carrega a tabela Bronze de Pedidos
try:
    df_bronze_pedidos = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada. Rode a Bronze primeiro!")

# 2. Isola o Micro-lote (Pega apenas o que ainda não foi processado na Silver)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    # Anti-Join: Mantém na Bronze apenas os IDs que NÃO estão na Silver
    df_micro_lote = df_bronze_pedidos.join(df_silver_atual, "id_pedido", "left_anti")
else:
    # Se a Silver não existe, o micro-lote é a Bronze inteira (Carga Histórica)
    df_micro_lote = df_bronze_pedidos

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote para processar: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência para os Joins (Evita quebrar se não existirem)
# =================================================================================
def obter_referencia(camada, tabela, colunas_select):
    if delta_existe(camada, tabela, STORAGE_OPTIONS):
        return ler_delta(camada, tabela, STORAGE_OPTIONS).select(*colunas_select).dropDuplicates()
    else:
        # Retorna DF vazio com o schema correto para o Join não falhar
        schema = StructType([StructField(c, StringType(), True) for c in colunas_select])
        return spark.createDataFrame([], schema)

# Vamos buscar os IDs nas tabelas Bronze ou Silver (depende de qual você já tem pronta)
# Renomeando as colunas logo na leitura para evitar ambiguidade no Join
df_clientes_ref = obter_referencia("bronze", "ecommerce_clientes", ["id_cliente"]) \
    .withColumnRenamed("id_cliente", "id_cliente_ref")

df_enderecos_ref = obter_referencia("bronze", "ecommerce_enderecos", ["id_endereco"]) \
    .withColumnRenamed("id_endereco", "id_endereco_ref")

# Para rastreamento, já estava correto:
if delta_existe("bronze", "ecommerce_rastreamento", STORAGE_OPTIONS):
    df_rastreamento_ref = ler_delta("bronze", "ecommerce_rastreamento", STORAGE_OPTIONS).select(
        # CORREÇÃO AQUI: O nome correto na origem é id_pedido_ecommerce
        F.col("id_pedido_ecommerce").alias("id_pedido_rastreio"), 
        F.col("status_entrega").alias("status_rastreamento")
    ).dropDuplicates(["id_pedido_rastreio"])
else:
    schema = StructType([
        StructField("id_pedido_rastreio", LongType(), True),
        StructField("status_rastreamento", StringType(), True)
    ])
    df_rastreamento_ref = spark.createDataFrame([], schema)

print("Tabelas de referência carregadas com sucesso.")
display(df_bronze_pedidos.limit(5))

## Aplicação das 10 Regras de Qualidade (Data Quality)


In [0]:
if qtd_novos > 0:
    # Listas de domínio
    status_validos = ["Processando", "Pagamento Aprovado", "Em Separação", "Enviado", "Entregue", "Cancelado"]
    metodos_validos = ["cartão de crédito", "pix", "boleto"]
    w_id_pedido = Window.partitionBy("id_pedido")

    # Prepara o DataFrame base tipado e faz os Joins
    df_base = df_micro_lote \
        .withColumn("valor_total_num", F.col("valor_total").cast("double")) \
        .withColumn("valor_frete_num", F.coalesce(F.col("valor_frete").cast("double"), F.lit(0.0))) \
        .withColumn("dt_pedido_ts", F.col("dt_pedido").cast("timestamp")) \
        .withColumn("dt_status_ts", F.col("dt_ultima_atualizacao_status").cast("timestamp")) \
        .withColumn("qtd_id_pedido", F.count("*").over(w_id_pedido)) \
        .join(df_clientes_ref, df_micro_lote.id_cliente == df_clientes_ref.id_cliente_ref, "left_outer") \
        .join(df_enderecos_ref, df_micro_lote.id_endereco_entrega == df_enderecos_ref.id_endereco_ref, "left_outer") \
        .join(df_rastreamento_ref, df_micro_lote.id_pedido == df_rastreamento_ref.id_pedido_rastreio, "left_outer")

    # Aplicação massiva de regras (Criando as flags booleanas Verdadeiro/Falso)
    df_silver_pedidos = df_base \
        .withColumn("r1_id_pedido_falhou", F.col("id_pedido").isNull() | (F.col("id_pedido").cast("string") == "") | (F.col("qtd_id_pedido") > 1)) \
        .withColumn("r2_id_cliente_fk_falhou", F.col("id_cliente").isNull() | (F.col("id_cliente").cast("string") == "") | df_clientes_ref.id_cliente_ref.isNull()) \
        .withColumn("r3_status_pedido_falhou", F.col("status_pedido").isNull() | (~F.col("status_pedido").isin(status_validos))) \
        .withColumn("r4_valor_total_falhou", F.col("valor_total_num").isNull() | (F.col("valor_total_num") <= 0)) \
        .withColumn("r5_metodo_pagamento_falhou", F.col("metodo_pagamento").isNull() | (~F.col("metodo_pagamento").isin(metodos_validos))) \
        .withColumn("r6_datas_status_falhou", F.col("dt_pedido_ts").isNull() | F.col("dt_status_ts").isNull() | (F.col("dt_status_ts") < F.col("dt_pedido_ts"))) \
        .withColumn("r7_frete_gratis_falhou", (F.col("valor_total_num") >= 250) & (F.col("valor_frete_num") > 0)) \
        .withColumn("r8_cancelado_rastreamento_falhou", (F.col("status_pedido") == "Cancelado") & F.col("status_rastreamento").isNotNull() & (F.col("status_rastreamento") != "Cancelado")) \
        .withColumn("r9_entrega_tempo_minimo_falhou", (F.col("status_pedido") == "Entregue") & (F.datediff(F.col("dt_status_ts"), F.col("dt_pedido_ts")) < 2)) \
        .withColumn("r10_endereco_fk_falhou", F.col("id_endereco_entrega").isNull() | (F.col("id_endereco_entrega").cast("string") == "") | df_enderecos_ref.id_endereco_ref.isNull())

    # Agrupa as regras Críticas para decidir quem passa para a Silver
    regras_criticas = [
        "r1_id_pedido_falhou", "r2_id_cliente_fk_falhou", "r3_status_pedido_falhou", 
        "r4_valor_total_falhou", "r5_metodo_pagamento_falhou", "r6_datas_status_falhou", 
        "r8_cancelado_rastreamento_falhou", "r10_endereco_fk_falhou"
    ]
    
    condicao_falha_critica = F.expr(" OR ".join(regras_criticas))

    df_silver_pedidos = df_silver_pedidos \
        .withColumn("silver_linha_valida", ~condicao_falha_critica) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))
    
    print("Regras de qualidade aplicadas com sucesso.")
else:
    print("Nenhum dado novo para aplicar regras.")

## Geração dos Logs (Data Quality Monitoring)

In [0]:
if qtd_novos > 0:
    regras_catalogo = [
        {"coluna": "r1_id_pedido_falhou", "regra": "R1_ID_PEDIDO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_ID_CLIENTE_FK_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_status_pedido_falhou", "regra": "R3_STATUS_PEDIDO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_valor_total_falhou", "regra": "R4_VALOR_TOTAL_INVALIDO", "severidade": "Critica"},
        {"coluna": "r5_metodo_pagamento_falhou", "regra": "R5_METODO_PAGAMENTO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r6_datas_status_falhou", "regra": "R6_DATA_STATUS_ANTERIOR_PEDIDO", "severidade": "Critica"},
        {"coluna": "r7_frete_gratis_falhou", "regra": "R7_FRETE_GRATIS_ACIMA_250", "severidade": "Aviso"},
        {"coluna": "r8_cancelado_rastreamento_falhou", "regra": "R8_CANCELADO_COM_RASTREAMENTO_ATIVO", "severidade": "Critica"},
        {"coluna": "r9_entrega_tempo_minimo_falhou", "regra": "R9_ENTREGA_TEMPO_MINIMO_INVALIDO", "severidade": "Aviso"},
        {"coluna": "r10_endereco_fk_falhou", "regra": "R10_ENDERECO_FK_INVALIDO", "severidade": "Critica"},
    ]

    total_registros = df_silver_pedidos.count()
    logs_list = []

    for r in regras_catalogo:
        qtd_falhas = df_silver_pedidos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID,
                TABELA_ALVO,
                r["regra"],
                "FAIL",
                r["severidade"],
                int(qtd_falhas),
                int(total_registros),
                datetime.now(timezone.utc),
                f"Bronze Delta ({TABELA_ALVO})"
            ))

    # Definimos o schema explicitamente aqui, já que não temos a função no utils
    schema_logs = StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema=schema_logs)
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema=schema_logs)

    print("Logs gerados:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    print("Etapa ignorada: não há micro-lote novo para gerar logs.")

# Gravação Final via SDK (Ignorando Unity Catalog)

In [0]:
if qtd_novos > 0:
    # ==============================================================================
    # 1. GRAVAÇÃO DA SILVER
    # ==============================================================================
    # Filtra as linhas boas e pega apenas as colunas originais do pedido + auditoria
    # Isso descarta colunas de erro e evita problemas com o PyArrow
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    df_silver_validos = df_silver_pedidos \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para gravar na Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos,
            camada="silver",
            tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=True 
        )
        if sucesso_silver:
            print("Tabela Silver de Pedidos atualizada com sucesso!")

    # ==============================================================================
    # 2. GRAVAÇÃO DOS LOGS DE DATA QUALITY
    # ==============================================================================
    # Se o DataFrame de logs novos estiver vazio (tudo passou na validação)
    if df_dq_monitoring_logs_novos.count() == 0:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    # Se a tabela já existir na raiz, aplicamos o filtro anti-join
    if delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        df_logs_historico = ler_delta(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS) \
            .filter(F.col("tabela") == TABELA_ALVO)
        
        condicao_join = [
            df_dq_monitoring_logs_novos.regra == df_logs_historico.regra,
            F.to_date(df_dq_monitoring_logs_novos.timestamp_execucao) == F.to_date(df_logs_historico.timestamp_execucao)
        ]
        
        df_logs_para_gravar = df_dq_monitoring_logs_novos.join(df_logs_historico, condicao_join, "left_anti")
        print(f"Filtro aplicado: {df_logs_para_gravar.count()} logs inéditos para gravação.")
    else:
        # Se a tabela NÃO existe, ela será criada do zero com o schema correto
        df_logs_para_gravar = df_dq_monitoring_logs_novos
        print("Tabela central de monitoramento não encontrada. Ela será criada nesta execução.")
        
    # Grava fisicamente na raiz do projeto se houver dados (ou se for a primeira criação)
    if df_logs_para_gravar.count() > 0 or not delta_existe(camada="", tabela=TABELA_DQ, storage_opts=STORAGE_OPTIONS):
        sucesso_logs = gravar_delta(
            df=df_logs_para_gravar,
            camada="", # Vazio para salvar diretamente na raiz do projeto
            tabela=TABELA_DQ,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade de Pedidos processados com sucesso na raiz!")
else:
    print("Rotina finalizada sem alterações físicas.")

# Geração dos logs de qualidade

In [0]:
if qtd_novos > 0:
    regras_catalogo = [
        {"coluna": "r1_id_pedido_falhou", "regra": "R1_ID_PEDIDO_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_id_cliente_fk_falhou", "regra": "R2_ID_CLIENTE_FK_INVALIDO", "severidade": "Critica"},
        {"coluna": "r3_status_pedido_falhou", "regra": "R3_STATUS_PEDIDO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r4_valor_total_falhou", "regra": "R4_VALOR_TOTAL_INVALIDO", "severidade": "Critica"},
        {"coluna": "r5_metodo_pagamento_falhou", "regra": "R5_METODO_PAGAMENTO_INVALIDO", "severidade": "Critica"},
        {"coluna": "r6_datas_status_falhou", "regra": "R6_DATA_STATUS_ANTERIOR_PEDIDO", "severidade": "Critica"},
        {"coluna": "r7_frete_gratis_falhou", "regra": "R7_FRETE_GRATIS_ACIMA_250", "severidade": "Aviso"},
        {"coluna": "r8_cancelado_rastreamento_falhou", "regra": "R8_CANCELADO_COM_RASTREAMENTO_ATIVO", "severidade": "Critica"},
        {"coluna": "r9_entrega_tempo_minimo_falhou", "regra": "R9_ENTREGA_TEMPO_MINIMO_INVALIDO", "severidade": "Aviso"},
        {"coluna": "r10_endereco_fk_falhou", "regra": "R10_ENDERECO_FK_INVALIDO", "severidade": "Critica"},
    ]

    total_registros = df_silver_pedidos.count()
    logs_list = []

    for r in regras_catalogo:
        qtd_falhas = df_silver_pedidos.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID,
                TABELA_ALVO,
                r["regra"],
                "FAIL",
                r["severidade"],
                int(qtd_falhas),
                int(total_registros),
                datetime.now(timezone.utc),
                f"Bronze Delta ({TABELA_ALVO})"
            ))

    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    print("Logs gerados:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    # A chamada aqui não vai mais dar erro porque a função existe no utils!
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Etapa ignorada: não há micro-lote novo para gerar logs.")

# Gravação da Silver válida e dos logs

In [0]:
if qtd_novos > 0:
    # 1. Filtra as linhas boas
    # 2. SEGREDO: Pega apenas as colunas originais do pedido + colunas de auditoria.
    # Isso joga fora as colunas com 'NullType' geradas pelas regras e evita o erro do PyArrow!
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    df_silver_validos = df_silver_pedidos \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Registros aprovados para gravar na Silver: {qtd_validos}")

    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos,
            camada="silver",
            tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=True 
        )
        if sucesso_silver:
            print("Tabela Silver atualizada com sucesso!")

    # Gravação dos logs usando a mesma função blindada
    if df_dq_monitoring_logs_novos.count() > 0:
        sucesso_logs = gravar_delta(
            df=df_dq_monitoring_logs_novos,
            camada="silver", 
            tabela="dq_monitoring_logs",
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False
        )
        if sucesso_logs:
            print("Logs de qualidade gravados na tabela de monitoramento!")
else:
    print("Rotina finalizada sem alterações físicas.")

## DIAGNÓSTICO E LIMPEZA DE LOGS DUPLICADOS

In [0]:
import pyspark.sql.functions as F

print("===== DIAGNÓSTICO E LIMPEZA DE LOGS DUPLICADOS =====")

# 1. Carrega a tabela atual do Data Lake
df_logs_atual = ler_delta("silver", "dq_monitoring_logs", STORAGE_OPTIONS)

# 2. Conta o total com e sem duplicados
total_antes = df_logs_atual.count()
total_unicos = df_logs_atual.dropDuplicates().count()

print(f"Total de registros na tabela: {total_antes}")
print(f"Total de registros únicos: {total_unicos}")

# 3. Se houver duplicidade, limpa e sobrescreve
if total_antes > total_unicos:
    qtd_duplicados = total_antes - total_unicos
    print(f"\nDetectamos {qtd_duplicados} logs repetidos! Iniciando limpeza...")
    
    # Cria um DataFrame apenas com linhas exclusivas
    df_logs_limpos = df_logs_atual.dropDuplicates()
    
    # Sobrescreve a tabela física (overwrite)
    sucesso = gravar_delta(
        df=df_logs_limpos,
        camada="silver",
        tabela="dq_monitoring_logs",
        storage_opts=STORAGE_OPTIONS,
        mode="overwrite", # <-- A MÁGICA ESTÁ AQUI
        particionar=False
    )
    
    if sucesso:
        print("Limpeza concluída com sucesso! Tabela de logs restaurada.")
else:
    print("\nA tabela está perfeitamente limpa. Não há repetições!")

# Validação final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

# 1. Validação da tabela Silver principal
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))
else:
    print(f"Silver {TABELA_ALVO} ainda não existe no Data Lake.")

# 2. Validação da tabela de Logs de Qualidade
if delta_existe("silver", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("silver", "dq_monitoring_logs", STORAGE_OPTIONS)
    
    # Filtra para mostrar apenas os logs referentes à tabela que acabou de rodar
    df_logs_filtrados = df_logs_validacao.filter(F.col("tabela") == TABELA_ALVO)
    
    print(f"Logs na dq_monitoring_logs para {TABELA_ALVO}:", df_logs_filtrados.count())
    display(df_logs_filtrados.orderBy(F.col("timestamp_execucao").desc()).limit(20))
else:
    print("Tabela dq_monitoring_logs ainda não existe no Data Lake.")